# mva_15-実践演習課題

- **学籍番号**:
- **氏名**:

---

このGoogle Colabファイルをコピーしてドライブに保存して下さい。<br>
ファイル名の学籍番号は自分の学籍番号に変更して下さい。<br>
必ずレポート末尾の『作成プロセスの報告』に分析や考察の作成手順を具体的に記述してください。

## **はじめに**

### **(1) 課題の目的**

本課題では、これまでの授業で習得した多変量解析（重回帰分析、クラスタ分析、主成分分析）を用い、データから意味のある解釈を導き出すプロセスを体験します。

### **(2) 演習のシナリオと必須分析項目**

あなたは、ある地域の「まちづくりコンサルタント」です。配布されたデータ（架空の50地区のデータ）を分析し、その地域の特徴を明らかにするとともに、住民の「地域満足度」を向上させるための提言をまとめてください。

**データの項目**

- ``Satisfaction``: 住民満足度（0-100） ※重回帰の目的変数
- ``Income``: 平均世帯年収
- ``Green_Area``: 公園・緑地面積
- ``Safety_Score``: 治安指標
- ``Convenience``: 商業施設の利便性
- ``Events``: 地域のイベント開催数
- ``Education``: 教育環境の充実度

#### **A. 重回帰分析**

- **目的**: 満足度を規定する要因を探る。
- **手法**: ``Satisfaction`` を目的変数、それ以外の6変数を説明変数として線形重回帰を行う。

**フォーム入力項目**:
- 決定係数 ($R^2$)
  - 回答形式: 小数点第3位を四捨五入して 第2位まで （例: ``0.83``）
- ``Income`` の偏回帰係数
  - 回答形式: 小数点第5位を四捨五入して 第4位まで （例: ``0.0305``）

#### **B. クラスタ分析**

- **目的**: 地区をタイプ分けする。
- **手法**: 全変数（``Satisfaction``含む）を標準化し、K-means法（クラスタ数 $K=3$） で分類する。必ず ``random_state=42`` を指定すること。

**フォーム入力項目**:
- 3つのクラスタのうち、**「Satisfactionの平均値が最も高いクラスタ」** を特定し、そのクラスタにおける ``Income`` の平均値 を回答してください。
  - 回答形式: 標準化前の元のスケールで、小数点第2位を四捨五入して 第1位まで （例: ``512.5``）。


#### **C. 主成分分析**

- **目的**: 地域の総合的な特徴を可視化する。
- **手法**: 説明変数（Income～Education）を標準化し、主成分分析を行う。

**フォーム入力項目**:
- 第2主成分までの 累積寄与率
  - 回答形式: 小数点第3位を四捨五入して 第2位まで （例: ``0.65``）


### **(3) 提出物**
演習に関する以下の項目をGoogle フォームに回答する形で提出して下さい。
- 必須分析項目（フォーム入力項目）の分析結果を数値で入力する。
- このGoogle Colab ファイルはドライブで「リンクを知っている全員が閲覧可」の設定にして共有リンクを入力する。
- このファイルを画面に映しながら、5分程度で口頭で分析結果について説明を行なった録画動画を、YouTubeに「限定公開」でアップロードし、そのリンクを入力。（「秀」評価希望者のみ）

### **(4) 評価基準**

- **正解率**: Googleフォームに入力された数値が、データから導かれる正しい値と一致しているか。
- **論理性**: レポートにおいて、数値結果から飛躍のない解釈ができているか。
- **独自性**: AI任せの一般的な記述ではなく、自分の理解と言葉でレポートを記述できているか。

# **データの入手と読み込み**

以下のGoogle Driveフォルダ（または共有リンク）にアクセスします。
- [mva_2025-KGU](https://drive.google.com/drive/folders/1S-34Y116zvv9LfwYL4EhMMy8B2k7B55m?usp=drive_link)
- 自分の学籍番号のファイル（例: 2024001.csv）を探します。
- ファイルを右クリックし、「リンクを取得」（または「共有」→「リンクをコピー」）を選択してURLをコピーしてください。

まずは配布された共有リンクから、自分の学籍番号のデータを読み込みます。

In [ ]:
# ==============================================================================
# はじめに必要なライブラリをimport して下さい。
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 日本語フォント対応（Colab用）
!pip install japanize-matplotlib
import japanize_matplotlib

# ==============================================================================
# 【設定】ここにあなたの学籍番号のCSVの共有リンク（URL）を貼り付けてください
# ==============================================================================
file_url = "https://drive.google.com/file/d/XXXXXXXXXXXXXX/view?usp=sharing"
# ==============================================================================

def read_csv_from_drive(url):
    """Google Driveの共有リンクから直接pandasデータフレームを読み込む関数"""
    try:
        if "drive.google.com" not in url:
            print("警告: Google Driveのリンクではありません。ローカルパスとして読み込みを試みます。")
            return pd.read_csv(url)

        file_id = url.split('/')[-2]
        dwn_url = f'https://drive.google.com/uc?id={file_id}'
        df = pd.read_csv(dwn_url)
        return df
    except Exception as e:
        print("エラー: データの読み込みに失敗しました。リンク権限などを確認してください。")
        print(e)
        return None

# データの読み込み
df = read_csv_from_drive(file_url)
display(df.head())

---
# **【実践演習課題】分析タスク**

### **A. 重回帰分析：満足度の要因を探る**

* **目的変数**: `Satisfaction`
* **説明変数**: `Income`, `Green_Area`, `Safety_Score`, `Convenience`, `Events`, `Education`

どの変数が満足度に大きく影響しているか（偏回帰係数）、モデルの精度はどれくらいか（決定係数）を確認して下さい。

In [ ]:
# ==============================================================================
# ここに分析コードを記載して下さい。
# ==============================================================================







**【考察】**
* （ここに分析結果から読み取れることを記述します。）

### **B. クラスタ分析：地区のタイプ分け**
* **手法**: K-means法
* **条件**: クラスタ数 $K=3$、`random_state=42`
* **前処理**: 全変数（ID除く）を標準化する

地区を3つのタイプに分類し、それぞれの特徴を把握します。


In [ ]:
# ==============================================================================
# ここに分析コードを記載して下さい。
# ==============================================================================











**【考察】**
* （ここにクラスタごとの特徴を記述します。）


### **C. 主成分分析：総合的な特徴の可視化**
* **手法**: PCA（主成分分析）
* **前処理**: 説明変数（Income～Education）のみを標準化して使用

変数を要約し、2次元の散布図上に地区をプロットして全体像を掴みます。

In [ ]:
# ==============================================================================
# ここに分析コードを記載して下さい。
# ==============================================================================










**【考察】**
* （ここにPC1、PC2の意味や、地区の分布についての考察を記述します。）

## **【必須】作成プロセスの報告**

本レポートの作成にあたり、どのようなリソース（授業資料、Webサイト、生成AI等）を参考にし、
どのような手順でコードの実装や考察の記述を行ったか、具体的に記述してください。

### **(1) コード作成のプロセス**
**記述欄:**
> ここに具体的に記述して下さい。

### **(2) 考察文章の作成プロセス**
**記述欄:**
> ここに具体的に記述して下さい。